# 02. Technical Indicators & Chart Analysis
Deep dive into computing and plotting RSI, MACD, Exponential Moving Averages (EMA 9, 21, 50, 200), Bollinger Bands, and Volume Spikes for any Binance symbol.

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backend.binance_client import binance_client
from backend.indicators import enrich_klines_dataframe

## 1. Fetch & Enrich Kline Data (ETH/USDT)

In [ ]:
symbol = 'ETHUSDT'
interval = '1h'
klines = binance_client.get_klines(symbol, interval=interval, limit=120)
df = enrich_klines_dataframe(klines)
df.tail(5)

## 2. Interactive Candlestick + RSI + MACD Subplots

In [ ]:
fig = make_subplots(
    rows=3, cols=1, 
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.6, 0.2, 0.2],
    subplot_titles=(f'{symbol} Price & EMA Overlay', 'RSI (14)', 'MACD (12, 26, 9)')
)

# Main Candlestick
fig.add_trace(go.Candlestick(
    x=df['open_time'],
    open=df['open'], high=df['high'], low=df['low'], close=df['close'],
    name='Candles'
), row=1, col=1)

# EMA 9 & EMA 21
fig.add_trace(go.Scatter(x=df['open_time'], y=df['ema_9'], name='EMA 9', line=dict(color='cyan', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df['open_time'], y=df['ema_21'], name='EMA 21', line=dict(color='orange', width=1)), row=1, col=1)

# RSI
fig.add_trace(go.Scatter(x=df['open_time'], y=df['rsi'], name='RSI', line=dict(color='purple', width=1.5)), row=2, col=1)
fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1)

# MACD
fig.add_trace(go.Scatter(x=df['open_time'], y=df['macd'], name='MACD', line=dict(color='blue', width=1)), row=3, col=1)
fig.add_trace(go.Scatter(x=df['open_time'], y=df['macd_signal'], name='Signal', line=dict(color='orange', width=1)), row=3, col=1)
colors = ['green' if val >= 0 else 'red' for val in df['macd_hist']]
fig.add_trace(go.Bar(x=df['open_time'], y=df['macd_hist'], name='Histogram', marker_color=colors), row=3, col=1)

fig.update_layout(height=800, title_text=f"{symbol} Technical Analysis View", template='plotly_dark')
fig.show()